# Dataset 1 — Appliances Energy Prediction (UCI)
### Etapa B — Python / Pandas

**Situação:** identificar períodos de consumo elevado dos eletrodomésticos e observar as condições de temperatura e umidade presentes nesses momentos.

**Entrada:** `amostra_dataset1_appliances.csv` — amostra de 10% exportada do Orange Data Mining (widget *Data Sampler* + *Save Data*), já com `Select Columns` aplicado (Appliances, lights, 3 atributos de temperatura, 3 de umidade).

In [3]:
import pandas as pd

df = pd.read_csv('amostra_dataset1_appliances.csv')

## 1. Inspeção inicial da amostra

In [4]:
df.head()

,Appliances,lights,T1,T3,T2,RH_1,RH_2,RH_3
0,40,0,20.8900,20.2900,17.7600,35.4000,39.1633,36.9000
1,90,10,21.8900,21.6333,21.2900,53.1000,45.3600,49.2267
2,50,0,21.3900,21.6667,17.6333,35.5000,40.5300,35.2000
3,50,0,21.3900,22.0333,23.8900,41.0333,34.8400,36.9333
4,70,0,19.9633,20.0000,16.4633,35.1267,40.1267,36.4000


In [5]:
df.shape

(1974, 8)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1974 entries, 0 to 1973
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Appliances  1974 non-null   int64  
 1   lights      1974 non-null   int64  
 2   T1          1974 non-null   float64
 3   T3          1974 non-null   float64
 4   T2          1974 non-null   float64
 5   RH_1        1974 non-null   float64
 6   RH_2        1974 non-null   float64
 7   RH_3        1974 non-null   float64
dtypes: float64(6), int64(2)
memory usage: 123.5 KB


In [7]:
df.describe()

,Appliances,lights,T1,T3,T2,RH_1,RH_2,RH_3
count,1974.000000,1974.000000,1974.000000,1974.000000,1974.000000,1974.000000,1974.000000,1974.000000
mean,93.044580,3.839919,21.664703,22.231615,20.293635,40.219193,40.411386,39.209733
std,93.826149,7.937076,1.574883,1.957275,2.126958,3.971923,4.037686,3.288818
min,20.000000,0.000000,16.790000,17.200000,16.100000,27.733300,24.063300,30.133300
25%,50.000000,0.000000,20.700000,20.700000,18.790000,37.205825,37.760000,36.900000
50%,60.000000,0.000000,21.600000,22.100000,20.000000,39.590000,40.433300,38.433300
75%,100.000000,0.000000,22.600000,23.290000,21.500000,42.991675,43.228325,41.741250
max,770.000000,50.000000,26.260000,29.198600,28.917500,57.496700,56.026700,49.226700


## 2. Renomear colunas
`Appliances` → `Consumo_Eletrodomesticos`, e simplificar pelo menos 3 atributos ambientais (aqui, os 6 de temperatura/umidade).

In [8]:
df = df.rename(columns={
    'Appliances': 'Consumo_Eletrodomesticos',
    'T1': 'Temp_Cozinha',
    'T2': 'Temp_Sala',
    'T3': 'Temp_Lavanderia',
    'RH_1': 'Umid_Cozinha',
    'RH_2': 'Umid_Sala',
    'RH_3': 'Umid_Lavanderia',
})

df.columns

Index(['Consumo_Eletrodomesticos', 'lights', 'Temp_Cozinha', 'Temp_Lavanderia',
       'Temp_Sala', 'Umid_Cozinha', 'Umid_Sala', 'Umid_Lavanderia'],
      dtype='object')

## 3. Maior consumo de eletrodomésticos registrado na amostra

In [9]:
consumo_max = df['Consumo_Eletrodomesticos'].max()
consumo_max

770

## 4. Limiar de 70% do máximo e DataFrame de consumo elevado

In [10]:
limiar_70 = 0.70 * consumo_max
limiar_70

539.0

In [11]:
df_consumo_alto = df[df['Consumo_Eletrodomesticos'] > limiar_70]
df_consumo_alto.head()

,Consumo_Eletrodomesticos,lights,Temp_Cozinha,Temp_Lavanderia,Temp_Sala,Umid_Cozinha,Umid_Sala,Umid_Lavanderia
352,620,0,24.3900,26.7300,25.3700,44.3333,37.7360,38.8633
427,650,0,20.8900,21.2000,18.2900,36.7000,39.6633,36.0000
453,560,30,21.3567,21.6333,19.5667,36.0900,37.1633,35.0300
616,570,10,24.8900,23.6000,23.5600,32.6667,31.1400,30.8233
696,590,30,19.8567,18.4267,18.7000,47.6633,34.1000,37.8333


## 5. Quantidade e percentual de registros acima do limiar

In [12]:
qtd_consumo_alto = df_consumo_alto.shape[0]
pct_consumo_alto = (qtd_consumo_alto / df.shape[0]) * 100

print(f"Registros de consumo elevado: {qtd_consumo_alto}")
print(f"Percentual da amostra: {pct_consumo_alto:.2f}%")

Registros de consumo elevado: 22
Percentual da amostra: 1.11%


## 6. Temperatura média (T1) e segundo DataFrame — consumo elevado E temperatura acima da média

In [13]:
temp_media = df['Temp_Cozinha'].mean()
temp_media

np.float64(21.664702634245188)

In [14]:
df_consumo_e_temp = df[
    (df['Consumo_Eletrodomesticos'] > limiar_70) &
    (df['Temp_Cozinha'] > temp_media)
]

qtd_consumo_e_temp = df_consumo_e_temp.shape[0]
pct_consumo_e_temp = (qtd_consumo_e_temp / df.shape[0]) * 100

print(f"Registros com consumo elevado e temperatura acima da média: {qtd_consumo_e_temp}")
print(f"Percentual da amostra: {pct_consumo_e_temp:.2f}%")

Registros com consumo elevado e temperatura acima da média: 8
Percentual da amostra: 0.41%


## 7. Comparação entre os dois DataFrames

| Critério | Quantidade | % da amostra |
|---|---|---|
| Só consumo > 70% do máximo | ver célula 5 | ver célula 5 |
| Consumo > 70% do máximo **e** temperatura > média | ver célula 6 | ver célula 6 |

Preencha a explicação abaixo com os números que você obteve ao rodar o notebook — não copie um texto genérico, o valor muda conforme a amostra (a amostragem é aleatória).

In [15]:
print(f"Só consumo elevado: {qtd_consumo_alto} registros ({pct_consumo_alto:.2f}%)")
print(f"Consumo elevado + temperatura acima da média: {qtd_consumo_e_temp} registros ({pct_consumo_e_temp:.2f}%)")
print(f"Diferença: {qtd_consumo_alto - qtd_consumo_e_temp} registros a menos ao acrescentar o critério de temperatura")

Só consumo elevado: 22 registros (1.11%)
Consumo elevado + temperatura acima da média: 8 registros (0.41%)
Diferença: 14 registros a menos ao acrescentar o critério de temperatura


**Interpretação esperada:** adicionar a condição de temperatura é uma interseção (`E` lógico), então o segundo DataFrame nunca pode ser maior que o primeiro — ele só pode manter ou reduzir a quantidade de registros. Se a redução for pequena, sugere que os picos de consumo já tendem a ocorrer em temperaturas mais altas. Se a redução for grande, os picos de consumo não estão fortemente associados a temperaturas altas — pode haver outro fator (uso noturno de eletrodomésticos, por exemplo) puxando o consumo para cima independentemente da temperatura.